# 01 - Visualising the three networks

Build the three reference networks at a small scale (`N = 20`, `k = 4`) so they are visually inspectable, and render them as both:

1. **Static** matplotlib figures (used in the report).
2. **Interactive** `pyvis` HTML pages (saved in `figures/`, openable in a browser).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

import matplotlib.pyplot as plt
import networkx as nx

from src.networks import build_all
from src.plotting import draw_networks_grid, to_pyvis

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

import matplotlib.pyplot as plt
import networkx as nx

from src.networks import build_all
from src.plotting import draw_networks_grid, to_pyvis

## 1. Build the three networks

We use the same `N` and `k` for all three networks. For the Watts–Strogatz graph we pick $\beta = 0.1$, which sits inside the small-world window we will explore in notebook 03.

In [ ]:
N, k, beta = 20, 4, 0.1
graphs = build_all(N=N, k=k, beta=beta, seed=42)

for name, G in graphs.items():
    avg_deg = 2 * G.number_of_edges() / G.number_of_nodes()
    print(f"{name:>4}: nodes={G.number_of_nodes()}, edges={G.number_of_edges()}, <deg>={avg_deg:.2f}")

## 2. Static comparison

Drawn side-by-side with matplotlib. We use a circular layout for the ring lattice and the Watts–Strogatz graph (so the rewired edges appear as chords across the circle) and a spring layout for Erdős–Rényi (where no spatial structure exists).

In [ ]:
fig = draw_networks_grid(graphs, save=FIGURES / "01_three_networks.png")
plt.show()

## 3. Interactive visualisations

`pyvis` renders each graph as an HTML page with draggable nodes and zoom. The files are written to `figures/`; open them in a browser to interact.

In [ ]:
layouts = {"ring": "circular", "er": "spring", "ws": "circular"}

for name, G in graphs.items():
    net = to_pyvis(G, layout=layouts[name])
    out = FIGURES / f"01_{name}.html"
    net.save_graph(str(out))
    print(f"saved {out.relative_to(ROOT)}")

## What to look for

- **Ring lattice**: perfectly symmetric, every node has exactly $k = 4$ neighbours. Walking from one node to its antipode takes $N/(2k) = 2.5$ hops on average.
- **Watts–Strogatz ($\beta = 0.1$)**: still mostly the ring, but a handful of edges have been rewired into long-range chords. Those few shortcuts are what collapse the diameter - the small-world effect.
- **Erdős–Rényi**: no spatial structure; edges scattered uniformly. Short paths but essentially zero local clustering.